# 第1讲：LLM基础与科研应用入门
## 用 LLaMA-3 对数字人文短视频转写文本做自动摘要

# 本讲定位

在数字人文（Digital Humanities）研究中，短视频、访谈录音、田野笔记等非结构化文本的规模远超人工处理能力。
**大语言模型**为我们提供了自动化"阅读"与"提炼"这些海量文本的新范式。

本讲是全部 10 讲的起点：
- 用最短路径加载 LLaMA-3，不纠缠复杂理论
- 以短视频转写文本为输入，完成**内容摘要生成**这一核心任务
- 代码聚焦核心步骤，规避复杂参数调试

> 你只需要 Python 基础 + Colab 免费 GPU，就能跑通全部代码。

# 本讲结构

## 1  环境准备与 GPU 自检
## 2  LLM 核心原理（简化版）
## 3  加载 LLaMA-3 8B（4位量化）
## 4  检查点：确认模型正常运行
## 5  准备短视频转写文本
## 6  生成内容摘要
## 7  结果展示与解读
## 快速回顾

# 透明度说明

> **模型来源**：本讲使用 `NousResearch/Meta-Llama-3-8B-Instruct`，这是 Meta 官方 Llama 3 权重由社区托管在 Hugging Face 上的镜像版本，无需 Hugging Face 账号或访问令牌即可直接下载。原始模型授权见 Meta Llama 3 Community License Agreement。
>
> **量化方案**：采用 4 位 NF4 量化将显存需求从 ~32 GB 压缩至 ~5-6 GB，使 Colab 免费 T4 GPU（16 GB）可顺畅运行。量化存在轻微精度损失，但对摘要任务的输出质量影响甚微。
>
> **数据伦理提示**：当你用真实访谈、田野录音等数据替换下方的占位文本时，请注意受访者的知情同意与数据敏感性——数字人文工作中，尊重研究对象比模型精度更重要。

# 第1章  环境准备与 GPU 自检

- 安装两个核心库：`transformers`（模型加载与推理）、`accelerate`（设备管理）
- 运行 GPU 自检——摘要推理在 CPU 上速度极慢，强烈建议使用 GPU

> **本地用户**：如果已在 `mllm-dh` conda 环境中运行，可注释掉 `!pip install` 行，直接执行 GPU 自检。

In [ ]:
# ===== 第1章：环境准备 =====
# Colab 用户：运行此单元格自动安装依赖。
# 本地 mllm-dh 环境用户：可注释掉 !pip 行，直接运行 GPU 自检。

!pip install -q -U transformers accelerate bitsandbytes

import torch

if not torch.cuda.is_available():
    print('未检测到 GPU。Colab 用户请选择：运行时 -> 更改运行时类型 -> T4 GPU。')
    print('CPU 可运行但速度很慢，建议连接 GPU 后重试。')
else:
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 已就绪：{device_name}')
    print(f'显存总量：{total_mem:.1f} GB')


# 第2章  LLM 核心原理（简化版）

## 大语言模型的"大"意味着什么？

LLaMA-3 8B 拥有 **80 亿个参数**（神经元连接权重），在约 15 万亿 token 的语料上训练而成。
它的核心架构是 **Transformer 解码器**，工作原理可以简化为三步：

1. **分词（Tokenization）**：将输入文本切分为词元（token），如"数字人文" -> ["数字", "人文"] 或更细的子词
2. **自注意力（Self-Attention）**：每个 token 查看上下文中的所有 token，决定"谁与谁相关"
3. **逐 token 预测（Autoregressive Generation）**：基于上文预测下一个最可能的 token，循环生成完整输出

# 第2章（续） 预训练 -> 指令微调

LLaMA-3 的训练分为两个阶段：

| 阶段 | 数据 | 目标 | 结果 |
|------|------|------|------|
| **预训练** | 海量网页、书籍、代码（15T tokens） | 学会"语言本身"——语法、事实、推理模式 | 能续写文本，但不一定听从指令 |
| **指令微调** | 人工标注的问答、摘要示例（百万级） | 学会"听懂人话"——按用户意图生成有用回答 | 能遵循指令完成特定任务 |

我们使用的 `Llama-3-8B-Instruct` 就是经过指令微调的版本，天然适合摘要、问答等任务。

> **为什么用 Instruct 版而非 Base 版？** Base 版只会续写，你如果输入"请总结下文"，它可能继续输出"请总结下文"后面的文本，而非给出摘要。Instruct 版经过对齐训练，能理解指令意图。

# 第3章  加载 LLaMA-3 8B（4位量化）

## 为什么要量化？

LLaMA-3 8B 的全精度权重（fp32）需要约 **32 GB** 显存——远超 Colab T4 的 16 GB。
**4 位 NF4 量化**把每个权重从 32 位压缩至 4 位，显存需求降至约 **5-6 GB**，精度损失有限。

本讲仅做**推理**（摘要生成），不做训练，所以只需加载量化模型即可，无需 LoRA 等微调组件。

In [ ]:
# ===== 第3章：加载分词器与量化模型 =====
# 首次运行需下载约 5 GB 权重，Colab 大约需要 3-5 分钟

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'NousResearch/Meta-Llama-3-8B-Instruct'

# 根据 GPU 能力自动选择计算精度
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'计算精度：{compute_dtype}')

# 4 位 NF4 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

# 加载分词器（Llama-3 默认没有 pad_token，必须显式设置）
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('分词器加载完成')

# 加载 4 位量化模型
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
print('模型加载完成')
print(f'参数量：{sum(p.numel() for p in model.parameters()) / 1e9:.2f} B')

# 第4章  检查点：确认模型正常运行

在输入真实的短视频文本之前，先用一个简单问题验证模型加载正确。

这是一个**有意设置的检查点**：如果模型权重损坏或量化配置不兼容，这里会立即报错——避免在后续的处理过程中浪费时间去排查基础问题。

In [ ]:
# ===== 检查点：确认模型正常运行 =====

test_prompt = '请用一句话解释什么是数字人文学。'

# Llama-3 Instruct 使用对话模板
messages = [{'role': 'user', 'content': test_prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(formatted, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== 检查点输出 ===')
print(response)
print('=' * 20)
print('模型加载正常，可以继续。')

# 第5章  准备短视频转写文本

以下是一段来自数字人文短视频的**模拟转写文本**（实际场景中可由 Whisper 等工具自动转录）。
内容包含古籍、版本学等专业术语，适合测试 LLM 的专业文本理解能力。

> **数据模板说明**：以下文本为占位示例。在真实使用中，将以下占位文本替换为你的短视频转录结果即可。
> 第6讲将介绍 Whisper 语音转写，实现从视频到文本的自动化。

In [ ]:
# ===== 第5章：加载转写文本 =====
# 模拟一段数字人文主题短视频的转写文本

TRANSCRIPT = (
    '大家好，今天我们来聊一聊宋代刻书的特点与版本鉴定的基本方法。'
    '宋代是中国雕版印刷的黄金时代，分为北宋刻本和南宋刻本两大系统。'
    '北宋刻本多用欧体字，笔画挺拔，纸墨精良。'
    '南宋刻本则有浙本、建本、蜀本三大地域流派，各有鲜明的版式风格。'
    '版本鉴定需要综合考察字体特征、版式布局、避讳字、刻工姓名以及纸张墨色等五个维度。'
    '例如，宋刻本常见的避讳方式是将当朝皇帝名讳减去末笔，这一特征可以作为断代的重要依据。'
    '此外，历代藏书家的批校题跋也是版本价值的重要参考。'
    '在数字化时代，我们还可以通过高精度扫描和深度学习技术来辅助识别刻本特征。'
    '希望这些内容能帮助大家初步了解古籍版本学的基础知识，谢谢。'
)

print(f'转写文本长度：{len(TRANSCRIPT)} 字符')
print()
print('--- 原文内容 ---')
print(TRANSCRIPT)

# 第6章  生成内容摘要

## 什么是好的摘要？

以本转录为例：
- **提取式摘要**：从原文中摘取关键句拼接，如"宋代刻本分北宋南宋，鉴定需考察字体、版式等五维度"
- **生成式摘要**：用自己的话重新组织，可能引入原文没有的概括性表述，如"视频系统介绍了宋代版本学的基本框架"

我们将用 LLaMA-3 Instruct 做**生成式摘要**，并通过系统提示指定输出要求：
- 控制在 100 字以内
- 保留关键术语（如"刻本"、"避讳"、"浙本/建本/蜀本"）
- 结构清晰，适合直接用于文献归档

In [ ]:
# ===== 第6章：生成摘要 =====

SYSTEM_PROMPT = (
    '你是一位数字人文领域的学者，擅长为古籍与文化遗产相关视频撰写精炼归档摘要。'
    '摘要要求：100字以内，保留关键术语，结构清晰，适合文献归档。'
)

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': f'请为以下短视频转写文本生成一份100字以内的归档摘要：\n\n{TRANSCRIPT}'},
]

formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

summary = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

print('===== 生成摘要 =====')
print(summary)
print('=' * 20)
print(f'摘要长度：{len(summary)} 字符')

# 第7章  结果展示与解读

## 对比：原文 vs. 摘要

一个有效的摘要应该满足三个标准：
1. **信息覆盖**：是否涵盖原文的核心要点？
2. **术语保留**："刻本"、"避讳"、"浙本"等专业术语是否准确出现？
3. **长度压缩**：是否将原文字数压缩至设定目标（100 字以内）？

以下代码将原文和摘要并列展示，方便人工评估。

In [ ]:
# ===== 第7章：对比展示 =====

print('===== 对比：原文 vs. 摘要 =====')
print()
print('【原文】')
print(TRANSCRIPT)
print()
print(f'  原文长度：{len(TRANSCRIPT)} 字符')
print()
print('【摘要】')
print(summary)
print()
print(f'  摘要长度：{len(summary)} 字符')
print(f'  压缩比：{len(summary) / len(TRANSCRIPT) * 100:.1f}%')
print()
print('--- 关键术语检查 ---')
for term in ['刻本', '避讳', '浙本', '建本', '蜀本', '版本鉴定', '雕版印刷']:
    in_original = term in TRANSCRIPT
    in_summary = term in summary
    status = 'OK' if in_summary else 'MISSING'
    print(f'  {term}: 原文{"有" if in_original else "无"} -> 摘要 {status}')

# 第7章（续） 摘要质量的参考维度

本讲仅进行人工评估。后续第8-9讲将引入 BLEU、ROUGE 等自动评估指标对摘要质量做量化分析。

| 维度 | 检查要点 |
|------|----------|
| **完整性** | 是否遗漏了宋代刻本两大系统（北宋/南宋）这一核心信息？ |
| **准确性** | 避讳方式"减去末笔"是否有误？浙本/建本/蜀本三个名称是否正确？ |
| **简洁性** | 摘要长度是否接近 100 字目标？是否有冗余表述？ |
| **可读性** | 句子是否通顺？是否有明显的不连贯或重复？ |

> **迭代建议**：如果摘要质量不理想，可以调整 `temperature`（越低越确定）、`max_new_tokens`（摘要长度上限）或优化系统提示词。

# 快速回顾（Quick Check）

**本讲完成了以下工作：**

- **环境搭建**：在 Colab 免费 GPU 上安装 transformers + accelerate，运行 GPU 自检
- **原理概览**：讲解了 Transformer 解码器核心三步骤（分词 -> 自注意力 -> 逐 token 预测）和预训练->指令微调两阶段
- **模型加载**：用 4 位 NF4 量化将 LLaMA-3 8B 的显存需求从 32 GB 降至 ~5-6 GB
- **摘要生成**：以短视频转写文本为输入，生成 100 字以内的归档摘要
- **结果评估**：通过原文-摘要对比和关键术语检查，初步评估摘要质量

> **和后续各讲的连贯性**：L01 让模型会"读"和"写"；L02 用 CLIP 把画面和文本对齐；L03-04 用 QLoRA 让模型真正认识数字人文术语。